# Random Forest: entrenamiento desde todas las sesiones

Este cuaderno corrige la lectura de 813 filas: parte de **`Registro_10min`**, que contiene 2.878 registros brutos de varias sesiones. Reconstruye las variables con rezagos y el objetivo a +10 minutos, sin modificar la aplicación original.

No todas las filas crudas serán aptas: se excluyen ventanas con lecturas Modbus inválidas, saltos temporales y escenarios con riego inicial heredado. Se informa un conteo por sesión y se genera `Base_guanabana_consolidada_para_RF.xlsx`.

Este cuaderno no tiene resultados precalculados. Solo deben informarse las métricas después de ejecutarlo con el Excel y verificar la independencia del conjunto de prueba.


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import math
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
VERSION_SIMULADOR = "balance_hidrico_temperatura_v2"
PASO_MINUTOS = 10
OBJETIVO = "HumedadObjetivo_10min"
VARIABLES_MODELO = [
    "Temperatura", "Precipitacion", "HumedadSuelo", "RiegoActivo",
    "HumedadSuelo_lag1", "HumedadSuelo_lag2",
    "Temperatura_lag1", "Temperatura_lag2",
    "Precipitacion_lag1", "Precipitacion_lag2",
    "RiegoActivo_lag1", "RiegoActivo_lag2",
]
CLAVES = ["Sesion", "Escenario", "TiempoSimuladoMin"]
ENTRADAS = VARIABLES_MODELO + [OBJETIVO, *CLAVES, "TiempoObjetivoMin"]


def elegir_excel(nombre="", usar_selector=False):
    """Busca el Excel en /content. No requiere la ventana de files.upload()."""
    carpeta = Path("/content") if Path("/content").exists() else Path.cwd()
    if usar_selector:
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("El selector de subida requiere Google Colab.") from exc
        print("Selecciona UN SOLO Excel (.xlsx).")
        archivos_subidos = files.upload()
        if len(archivos_subidos) != 1:
            raise ValueError("Debes subir exactamente un archivo .xlsx.")
        return Path(next(iter(archivos_subidos))).resolve()

    if nombre:
        archivo = Path(nombre)
        if not archivo.is_absolute():
            archivo = carpeta / archivo
        if not archivo.is_file():
            raise FileNotFoundError(
                f"No se encontró {archivo}. Sube el Excel desde el icono de carpeta "
                "del panel izquierdo de Colab y ejecútalo de nuevo."
            )
        return archivo

    encontrados = sorted(carpeta.glob("*.xlsx"), key=lambda x: x.stat().st_mtime, reverse=True)
    if not encontrados:
        raise FileNotFoundError(
            "No se encontró ningún Excel (.xlsx) en /content. "
            "En Google Colab, abre el icono de carpeta del panel izquierdo, "
            "pulsa Subir y escoge Base_guanabana_consolidada_para_RF.xlsx. "
            "Después ejecuta de nuevo esta celda."
        )
    if len(encontrados) > 1:
        print("ATENCIÓN: hay varios Excel en /content; se seleccionará el más reciente.")
        for a in encontrados:
            print(" -", a.name)
    return encontrados[0]



def construir_desde_registro_completo(archivo):
    """Reconstruye X(t), y(t+10) por sesion y escenario desde Registro_10min.

    Excluye ventanas con saltos temporales, Modbus invalido y escenarios
    cuyo inicio de riego fue heredado, igual que el exportador original.
    """
    columnas = [
        "Sesion", "Escenario", "TiempoSimuladoMin", "Temperatura",
        "Precipitacion", "HumedadSuelo", "RiegoActivo",
        "ModbusConectado", "LecturaRiegoValida",
        "InicioRiegoHeredado", "ModeloSueloVersion",
    ]
    raw = pd.read_excel(archivo, sheet_name="Registro_10min", engine="openpyxl")
    faltantes = [c for c in columnas if c not in raw.columns]
    if faltantes:
        raise ValueError("Faltan columnas en Registro_10min: " + str(faltantes))
    print("Filas originales en Registro_10min:", len(raw))
    raw = raw[columnas].copy()
    raw["Sesion"] = raw["Sesion"].astype("string").str.strip()
    raw["ModeloSueloVersion"] = raw["ModeloSueloVersion"].astype("string").str.strip()
    cols_num = [c for c in columnas if c not in ("Sesion", "ModeloSueloVersion")]
    raw[cols_num] = raw[cols_num].apply(pd.to_numeric, errors="coerce")
    inicio = len(raw)
    raw = raw.replace([np.inf, -np.inf], np.nan).dropna(subset=columnas)
    raw = raw.loc[raw["Sesion"].str.len().gt(0)].copy()
    if (raw["ModeloSueloVersion"] != VERSION_SIMULADOR).any():
        raise ValueError("Se encontraron versiones de simulador incompatibles.")
    raw = raw.loc[
        raw["Escenario"].ge(1)
        & np.isclose(raw["Escenario"], np.round(raw["Escenario"]))
        & raw["HumedadSuelo"].between(0, 100)
        & raw["Precipitacion"].ge(0)
        & raw["RiegoActivo"].isin([0, 1])
        & raw["ModbusConectado"].isin([0, 1])
        & raw["LecturaRiegoValida"].isin([0, 1])
        & raw["InicioRiegoHeredado"].isin([0, 1])
    ].copy()
    raw["Escenario"] = raw["Escenario"].astype(int)
    raw = raw.sort_values(["Sesion", "Escenario", "TiempoSimuladoMin"])
    antes_unicos = len(raw)
    raw = raw.drop_duplicates(
        ["Sesion", "Escenario", "TiempoSimuladoMin"], keep="last"
    )
    print("Filas descartadas por campos incorrectos:", inicio - antes_unicos)
    print("Marcas temporales duplicadas:", antes_unicos - len(raw))

    filas = []
    resumen = []
    for (sesion, escenario), grupo in raw.groupby(["Sesion", "Escenario"], sort=True):
        grupo = grupo.sort_values("TiempoSimuladoMin").reset_index(drop=True)
        heredado = bool((grupo["InicioRiegoHeredado"] == 1).any())
        aptas, saltos, modbus_invalido = 0, 0, 0
        if not heredado:
            for i in range(2, len(grupo) - 1):
                bloque = grupo.iloc[i-2:i+2]
                tiempos = bloque["TiempoSimuladoMin"].to_numpy(dtype=float)
                if not np.allclose(np.diff(tiempos), PASO_MINUTOS, atol=1e-6):
                    saltos += 1
                    continue
                if not (
                    bloque["ModbusConectado"].eq(1)
                    & bloque["LecturaRiegoValida"].eq(1)
                ).all():
                    modbus_invalido += 1
                    continue
                lag2, lag1, actual, futuro = [
                    bloque.iloc[j] for j in range(4)
                ]
                fila = {
                    "Temperatura": float(actual["Temperatura"]),
                    "Precipitacion": float(actual["Precipitacion"]),
                    "HumedadSuelo": float(actual["HumedadSuelo"]),
                    "RiegoActivo": float(actual["RiegoActivo"]),
                }
                for variable in [
                    "HumedadSuelo", "Temperatura", "Precipitacion", "RiegoActivo"
                ]:
                    fila[variable + "_lag1"] = float(lag1[variable])
                    fila[variable + "_lag2"] = float(lag2[variable])
                fila[OBJETIVO] = float(futuro["HumedadSuelo"])
                fila.update({
                    "Sesion": str(sesion),
                    "Escenario": int(escenario),
                    "TiempoSimuladoMin": float(actual["TiempoSimuladoMin"]),
                    "TiempoObjetivoMin": float(futuro["TiempoSimuladoMin"]),
                    "ModeloSueloVersion": VERSION_SIMULADOR,
                })
                filas.append(fila)
                aptas += 1
        resumen.append({
            "Sesion": str(sesion),
            "Escenario": int(escenario),
            "MuestrasRegistro": len(grupo),
            "FilasEntrenamiento": aptas,
            "RiegoInicioHeredado": int(heredado),
            "VentanasConSaltos": saltos,
            "VentanasModbusInvalido": modbus_invalido,
        })
        print(
            f"Sesion={sesion} escenario={escenario}: "
            f"{len(grupo)} registros -> {aptas} muestras; "
            f"saltos={saltos}, modbus_invalido={modbus_invalido}, "
            f"riego_inicio_heredado={heredado}"
        )

    if not filas:
        raise ValueError(
            "No hay ventanas validas para entrenar. "
            "Revisa continuidad de tiempo y estado de Modbus."
        )

    construido = pd.DataFrame(filas)
    construido = construido[
        VARIABLES_MODELO + [OBJETIVO, "Sesion", "Escenario",
                            "TiempoSimuladoMin", "TiempoObjetivoMin",
                            "ModeloSueloVersion"]
    ].sort_values(CLAVES).reset_index(drop=True)
    print(
        "\nMuestras aptas totales:", len(construido),
        "| Escenarios:", construido.groupby(["Sesion", "Escenario"]).ngroups
    )

    salida = Path("/content/Base_guanabana_consolidada_para_RF.xlsx")
    with pd.ExcelWriter(salida, engine="openpyxl") as writer:
        construido.to_excel(writer, sheet_name="Entrenamiento_RF", index=False)
        raw.to_excel(writer, sheet_name="Registro_10min", index=False)
        pd.DataFrame(resumen).to_excel(
            writer, sheet_name="ResumenEscenarios", index=False
        )
        pd.read_excel(archivo, sheet_name="Parametros", engine="openpyxl").to_excel(
            writer, sheet_name="Parametros", index=False
        )
    print("Excel consolidado para publicar en GitHub:", salida)
    return construido


def cargar_y_validar(archivo):
    archivo = Path(archivo)
    if archivo.suffix.lower() != ".xlsx":
        raise ValueError("El archivo seleccionado debe ser .xlsx, no .csv.")

    parametros = pd.read_excel(archivo, sheet_name="Parametros", engine="openpyxl")
    if not {"Parametro", "Valor"}.issubset(parametros.columns):
        raise ValueError("La hoja Parametros debe contener las columnas Parametro y Valor.")
    cfg = dict(zip(parametros["Parametro"].astype(str).str.strip(),
                   parametros["Valor"].astype(str).str.strip()))
    version = cfg.get("ModeloSueloVersion", cfg.get("Version del simulador"))
    if version != VERSION_SIMULADOR:
        raise ValueError(
            f"Versión incompatible ({version}). Se requiere {VERSION_SIMULADOR}."
        )
    paso = cfg.get("Paso_muestreo_min")
    if paso is not None and int(float(paso)) != PASO_MINUTOS:
        raise ValueError("La base no corresponde a intervalos de 10 minutos simulados.")

    datos = construir_desde_registro_completo(archivo)
    faltantes = [c for c in ENTRADAS if c not in datos.columns]
    if faltantes:
        raise ValueError(f"La hoja Entrenamiento_RF no contiene: {faltantes}")
    if "ModeloSueloVersion" in datos.columns:
        versiones = set(datos["ModeloSueloVersion"].dropna().astype(str).str.strip())
        if versiones and versiones != {VERSION_SIMULADOR}:
            raise ValueError(f"Hay registros de otras versiones: {versiones}")

    n_original = len(datos)
    datos = datos[ENTRADAS].copy()
    datos["Sesion"] = datos["Sesion"].astype("string").str.strip()
    for c in VARIABLES_MODELO + [OBJETIVO, "Escenario", "TiempoSimuladoMin", "TiempoObjetivoMin"]:
        datos[c] = pd.to_numeric(datos[c], errors="coerce")
    datos = datos.replace([np.inf, -np.inf], np.nan).dropna(subset=ENTRADAS)
    datos = datos.loc[datos["Sesion"].str.len() > 0].copy()
    datos = datos.loc[datos["Escenario"].ge(1) &
                      np.isclose(datos["Escenario"], np.round(datos["Escenario"]))].copy()
    datos = datos.loc[np.isclose(datos["TiempoSimuladoMin"] % PASO_MINUTOS, 0)].copy()
    datos = datos.loc[np.isclose(
        datos["TiempoObjetivoMin"] - datos["TiempoSimuladoMin"], PASO_MINUTOS
    )].copy()
    for c in ["HumedadSuelo", "HumedadSuelo_lag1", "HumedadSuelo_lag2", OBJETIVO]:
        datos = datos.loc[datos[c].between(0, 100)].copy()
    for c in ["Precipitacion", "Precipitacion_lag1", "Precipitacion_lag2"]:
        datos = datos.loc[datos[c].ge(0)].copy()
    for c in ["RiegoActivo", "RiegoActivo_lag1", "RiegoActivo_lag2"]:
        datos = datos.loc[datos[c].isin([0, 1])].copy()

    datos["Escenario"] = datos["Escenario"].astype(int)
    datos = datos.sort_values(CLAVES).drop_duplicates(CLAVES, keep="last")
    datos = datos.reset_index(drop=True)
    if len(datos) < 100:
        raise ValueError(f"Hay solo {len(datos)} filas válidas; se necesitan al menos 100.")
    print(f"Archivo: {archivo.name}")
    print(f"Muestras de la hoja Entrenamiento_RF: {n_original}")
    print(f"Muestras válidas, sin duplicados: {len(datos)}")
    print(f"Muestras descartadas: {n_original - len(datos)}")
    print(f"Escenarios independientes: {datos.groupby(['Sesion', 'Escenario']).ngroups}")
    return datos


def separar_sin_fuga(datos):
    """Dejar fuera escenarios completos cuando sea posible."""
    sizes = datos.groupby(["Sesion", "Escenario"], sort=True).size()
    n = len(datos)
    if len(sizes) >= 2:
        elegidos, cantidad = [], 0
        for k in reversed(list(sizes.index)):
            nuevo = cantidad + int(sizes[k])
            if n - nuevo < 100:
                break
            elegidos.append(k)
            cantidad = nuevo
            if cantidad >= 0.15 * n:
                break
        if elegidos and cantidad >= 20:
            idx = pd.MultiIndex.from_frame(datos[["Sesion", "Escenario"]])
            mascara = idx.isin(elegidos)
            train, test = datos.loc[~mascara].copy(), datos.loc[mascara].copy()
            print("División por escenarios completos, sin mezclar registros del mismo escenario.")
            print("Escenarios de prueba:", list(reversed(elegidos)))
            return train, test, "escenarios"

    # Esta evaluación es solo exploratoria si existe un escenario único.
    corte = int(n * 0.80)
    train, test = datos.iloc[:max(0, corte - 3)].copy(), datos.iloc[corte:].copy()
    if len(train) < 50 or len(test) < 20:
        raise ValueError("No fue posible obtener suficientes datos de entrenamiento y prueba.")
    print("ATENCIÓN: un solo escenario útil; división cronológica con 3 filas de separación.")
    print("El resultado es preliminar, NO sirve para aprobar el modelo final.")
    return train, test, "temporal_preliminar"


def entrenar_y_exportar(datos, directorio=None, mostrar_graficas=True):
    carpeta = Path(directorio) if directorio else (Path('/content') if Path('/content').exists() else Path.cwd())
    carpeta.mkdir(parents=True, exist_ok=True)
    train, test, metodo = separar_sin_fuga(datos)
    X_train, y_train = train[VARIABLES_MODELO], train[OBJETIVO]
    X_test, y_test = test[VARIABLES_MODELO], test[OBJETIVO]
    print(f"Entrenamiento: {len(train)} filas | Prueba: {len(test)} filas")
    print("Entrenando Random Forest con 300 árboles...")
    modelo = RandomForestRegressor(
        n_estimators=300, max_depth=15, min_samples_split=4,
        min_samples_leaf=2, random_state=42, n_jobs=-1,
    )
    modelo.fit(X_train, y_train)
    pred = np.clip(modelo.predict(X_test), 0, 100)
    pred_base = X_test["HumedadSuelo"].to_numpy()
    mae = float(mean_absolute_error(y_test, pred))
    rmse = float(math.sqrt(mean_squared_error(y_test, pred)))
    r2 = float(r2_score(y_test, pred))
    mae_base = float(mean_absolute_error(y_test, pred_base))
    rmse_base = float(math.sqrt(mean_squared_error(y_test, pred_base)))
    aprobado = metodo == "escenarios" and rmse < rmse_base and r2 > 0
    metricas = {
        "MAE": mae, "RMSE": rmse, "R2": r2,
        "MAE_referencia": mae_base, "RMSE_referencia": rmse_base,
        "registros_entrenamiento": len(X_train), "registros_prueba": len(X_test),
        "metodo_division": metodo, "escenarios_totales": int(datos.groupby(["Sesion", "Escenario"]).ngroups),
        "supera_referencia": bool(rmse < rmse_base),
        "apto_para_prueba_en_aplicacion": bool(aprobado),
    }
    print("\n=== RESULTADOS (humedad a +10 minutos simulados) ===")
    print(f"MAE RF: {mae:.4f} | RMSE RF: {rmse:.4f} | R²: {r2:.4f}")
    print(f"MAE referencia: {mae_base:.4f} | RMSE referencia: {rmse_base:.4f}")
    print("VALIDADO para prueba en aplicación" if aprobado else
          "NO VALIDADO: no reemplaces el modelo actual de la aplicación")

    paquete = {
        "modelo": modelo, "variables": VARIABLES_MODELO, "paso_minutos": PASO_MINUTOS,
        "lags": 2, "simulador_modelo_version": VERSION_SIMULADOR,
        "sklearn_version": sklearn.__version__,
        "fecha_entrenamiento": datetime.now().isoformat(timespec="seconds"),
        "objetivo": "Humedad del suelo a los siguientes 10 minutos simulados",
        "metricas_prueba": metricas,
    }
    nombre_modelo = ("modelo_prediccion_humedad.pkl" if aprobado
                     else "modelo_prediccion_humedad_NO_VALIDADO.pkl")
    ruta_modelo = carpeta / nombre_modelo
    joblib.dump(paquete, ruta_modelo)
    columnas_salida = ["Sesion", "Escenario", "TiempoSimuladoMin", "HumedadSuelo", OBJETIVO]
    resultado = test[columnas_salida].copy()
    resultado["HumedadPredicha_RF"] = pred
    resultado["PrediccionReferencia"] = pred_base
    ruta_csv = carpeta / "resultados_prueba_random_forest.csv"
    resultado.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
    ruta_json = carpeta / "metricas_random_forest.json"
    ruta_json.write_text(json.dumps(metricas, indent=2, ensure_ascii=False), encoding="utf-8")
    ruta_zip = carpeta / "resultados_RF_guanabana.zip"
    with zipfile.ZipFile(ruta_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
        archivos_zip = [ruta_modelo, ruta_csv, ruta_json]
        excel_consolidado = Path("/content/Base_guanabana_consolidada_para_RF.xlsx")
        if excel_consolidado.is_file():
            archivos_zip.append(excel_consolidado)
        for f in archivos_zip:
            z.write(f, arcname=f.name)

    if mostrar_graficas:
        plt.figure(figsize=(11, 4))
        plt.plot(y_test.to_numpy(), label="Humedad real a +10 min")
        plt.plot(pred, label="Predicción Random Forest")
        plt.title("Humedad real vs. predicha (prueba)")
        plt.xlabel("Registro de prueba")
        plt.ylabel("Agua disponible (%)")
        plt.legend()
        plt.tight_layout()
        plt.show()
        imp = pd.Series(modelo.feature_importances_, index=VARIABLES_MODELO).sort_values()
        plt.figure(figsize=(9, 5))
        plt.barh(imp.index, imp.values)
        plt.title("Importancia de variables")
        plt.tight_layout()
        plt.show()

    print("\nArchivos generados:")
    for a in [ruta_modelo, ruta_csv, ruta_json, ruta_zip]:
        print(" -", a)
    print("Si el navegador bloquea la descarga, abre Archivos a la izquierda en Colab,")
    print("pulsa los tres puntos junto al archivo y elige Descargar.")
    return {"modelo": ruta_modelo, "csv": ruta_csv, "metricas": ruta_json, "zip": ruta_zip, "aprobado": aprobado}

In [ ]:
# Descargar automaticamente el Excel ORIGINAL desde el repositorio.
# Se usa el commit donde se publico el archivo, para fijar una version reproducible.
from urllib.request import urlretrieve
URL_EXCEL = (
    "https://raw.githubusercontent.com/pealvaga/"
    "riego-inteligente-guanabana-simulacion/"
    "4e46683032f2894c2d1f13539403c0e5d1c3888c/"
    "06_Datos/Datos_entrenamiento_guanabana_20261110_103013.xlsx"
)
archivo_excel = Path("/content/Datos_entrenamiento_guanabana_20261110_103013.xlsx")
try:
    if not archivo_excel.exists():
        urlretrieve(URL_EXCEL, archivo_excel)
except Exception as exc:
    print("Descarga automatica no disponible:", exc)
    print("Descarga el Excel de 06_Datos en GitHub y subelo manualmente a Colab.")
    from google.colab import files
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError("Sube un solo Excel .xlsx")
    archivo_excel = Path(next(iter(subidos))).resolve()
datos = cargar_y_validar(archivo_excel)
print("\nTabla consolidada lista para el entrenamiento.")


In [ ]:
# Entrena con las muestras reconstruidas de todas las sesiones.
# Comparar siempre con persistencia y comprobar prueba independiente.
archivos_generados = entrenar_y_exportar(datos, directorio="/content")

In [ ]:
# Ejecutar esta celda SOLO para descargar, después de entrenar.
# Descarga tambien Base_guanabana_consolidada_para_RF.xlsx desde Colab.
from google.colab import files
archivo_zip = archivos_generados["zip"]
print("Descargando:", archivo_zip.name)
files.download(str(archivo_zip))
# Si el navegador bloquea esta ventana: panel Archivos (izquierda)
# → resultados_RF_guanabana.zip → tres puntos → Descargar.